# Fraunhofer Diffraction Simulator

Interactive 2D simulator for **slit**, **rectangular**, **circular**, and coherent **multiple/mixed** apertures (Taller 4, exercise 20).

The implementation follows the analytical Fourier-transform relations in [`theory and context/Fraunhofer_Transfomada_Fourier_espacial.pdf`](theory%20and%20context/Fraunhofer_Transfomada_Fourier_espacial.pdf):

- Rectangle/slit: $A(f_x,f_y)=ab\,\mathrm{sinc}(af_x)\,\mathrm{sinc}(bf_y)$.
- Circle: $A(f_r)=\pi r^2\,2J_1(2\pi r f_r)/(2\pi r f_r)$.
- Translated coherent openings: amplitudes acquire $e^{-i2\pi(f_xx_0+f_yy_0)}$ and are summed before taking $|A|^2$.
- Screen mapping: $f_x=x'/(\lambda z)$ and $f_y=y'/(\lambda z)$, where $\lambda=\lambda_0/n$.

Before a grid or field is evaluated, the simulator enforces $N_F=R_{\max}^2/(\lambda z)\le N_{F,\max}$. The default $N_{F,\max}=0.1$ is an explicit interpretation of the source's far-field requirement; it can be made stricter in **Advanced controls**.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import sys

import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display

import dashboard_tools as dashboard
import diffraction_config as config
import diffraction_engine as engine
import screen_tools as screen

print("Python:", sys.executable)
for module in (np, plt.matplotlib, widgets):
    print(f"  {module.__name__}: {module.__version__}")
print("Diffraction engine + screen tools loaded OK.")

## 1. Analytical sanity checks

These checks validate the implementation before the interactive UI is constructed:

1. a rectangular/slit transform vanishes at $f_x=1/a$;
2. the circular transform vanishes at the first zero of $J_1$;
3. changing refractive index changes the in-medium wavelength as $\lambda_0/n$;
4. an invalid near-field configuration is rejected by the mandatory gate.

In [ ]:
from scipy.special import jn_zeros

lambda_0 = config.WAVELENGTH_NM.to_si(config.WAVELENGTH_NM.default)
slit = engine.Aperture.slit(
    config.DEFAULT_SLIT_WIDTH_M,
    config.DEFAULT_SLIT_LENGTH_M,
)
circle = engine.Aperture.circle(config.DEFAULT_CIRCLE_RADIUS_M)

# Rectangle/slit first transform zero.
fx = np.array([0.0, 1.0 / slit.width])
slit_field = engine.aperture_amplitude(fx, np.zeros_like(fx), slit)
np.testing.assert_allclose(slit_field[0], slit.area)
np.testing.assert_allclose(slit_field[1], 0.0, atol=slit.area * 1e-12)

# Circular first transform zero: 2*pi*r*fr is the first root of J1.
first_j1_root = jn_zeros(1, 1)[0]
fr_zero = first_j1_root / (2.0 * np.pi * circle.radius)
circle_zero = engine.aperture_amplitude(fr_zero, 0.0, circle)
np.testing.assert_allclose(circle_zero, 0.0, atol=circle.area * 1e-12)

# Wavelength in a medium.
np.testing.assert_allclose(
    engine.medium_wavelength(lambda_0, config.REFRACTIVE_INDEX.maximum),
    lambda_0 / config.REFRACTIVE_INDEX.maximum,
)

# The gate must reject a distance below the reported minimum.
probe = engine.evaluate_far_field(
    [slit], lambda_0, config.DISTANCE_M.default,
    max_fresnel_number=config.MAX_FRESNEL_NUMBER.default,
)
try:
    engine.require_far_field(
        [slit], lambda_0, probe.required_distance / 2.0,
        max_fresnel_number=config.MAX_FRESNEL_NUMBER.default,
    )
except engine.FarFieldError:
    pass
else:
    raise AssertionError("The far-field gate accepted an invalid configuration.")

print("All analytical sanity checks passed.")

## 2. Interactive dashboard

The simulator now behaves like a compact optical workbench rather than a graph-only notebook:

1. The top schematic shows the source, aperture plane, propagation distance, and observation plane.
2. The center dashboard keeps the controls, aperture shape, and wavelength-colored diffraction camera visible together.
3. Choose **Slit**, **Rectangle**, **Circle**, or **Multiple** with the case buttons. Mixed mode supports independently sized, positioned, and oriented openings.
4. **Auto update** refreshes after sliders are released; use **Refresh** at any time or **Reset** to restore the configured defaults.
5. Optional **Profiles** and **Detailed geometry** views remain available below the visual dashboard.

The aperture and optical schematic are always shown. If the far-field check fails, the program does not calculate Fraunhofer diffraction; the observation panel instead reports the minimum valid distance while leaving the selected optical setup visible.

In [ ]:
def linked_float(spec, description, value=None):
    """Create linked slider/text controls from centralized configuration."""
    selected = spec.default if value is None else value
    slider = widgets.FloatSlider(
        min=spec.minimum,
        max=spec.maximum,
        step=spec.step,
        value=selected,
        description=description,
        readout=False,
        continuous_update=False,
        style={"description_width": "initial"},
        layout=widgets.Layout(width=config.SLIDER_WIDTH),
    )
    text = widgets.BoundedFloatText(
        min=spec.minimum,
        max=spec.maximum,
        step=spec.step,
        value=selected,
        layout=widgets.Layout(width=config.TEXT_WIDTH),
    )
    widgets.link((slider, "value"), (text, "value"))
    return slider, widgets.HBox([slider, text])


def linked_int(spec, description):
    slider = widgets.IntSlider(
        min=int(spec.minimum),
        max=int(spec.maximum),
        step=int(spec.step),
        value=int(spec.default),
        description=description,
        readout=False,
        continuous_update=False,
        style={"description_width": "initial"},
        layout=widgets.Layout(width=config.SLIDER_WIDTH),
    )
    text = widgets.BoundedIntText(
        min=int(spec.minimum),
        max=int(spec.maximum),
        step=int(spec.step),
        value=int(spec.default),
        layout=widgets.Layout(width=config.TEXT_WIDTH),
    )
    widgets.link((slider, "value"), (text, "value"))
    return slider, widgets.HBox([slider, text])


def bounded_float(spec, description, value=None):
    return widgets.BoundedFloatText(
        min=spec.minimum,
        max=spec.maximum,
        step=spec.step,
        value=spec.default if value is None else value,
        description=description,
        style={"description_width": "initial"},
    )


case = widgets.Dropdown(options=config.CASE_OPTIONS, value="slit", description="Case")
slit_orientation = widgets.ToggleButtons(
    options=config.SLIT_ORIENTATION_OPTIONS,
    value="vertical",
    description="Slit orientation",
    style={"description_width": "initial"},
)
view_buttons = {}
for view_label, view_key, view_icon in config.VIEW_OPTIONS:
    view_buttons[view_key] = widgets.ToggleButton(
        value=view_key in config.DEFAULT_ACTIVE_VIEWS,
        description=view_label,
        icon=view_icon,
        tooltip=f"Show or hide {view_label.lower()}",
    )
view_selector = widgets.HBox(tuple(view_buttons.values()))

wavelength, wavelength_box = linked_float(config.WAVELENGTH_NM, "lambda_0 (nm)")
refractive_index, refractive_index_box = linked_float(config.REFRACTIVE_INDEX, "n")
distance, distance_box = linked_float(config.DISTANCE_M, "distance z (m)")
screen_half_width, screen_half_width_box = linked_float(
    config.SCREEN_HALF_WIDTH_MM, "screen half-width (mm)"
)
zoom, zoom_box = linked_float(config.ZOOM, "zoom")
resolution, resolution_box = linked_int(config.RESOLUTION, "resolution")
max_fresnel, max_fresnel_box = linked_float(
    config.MAX_FRESNEL_NUMBER, "maximum N_F"
)

slit_width, slit_width_box = linked_float(
    config.APERTURE_WIDTH_UM,
    "slit width (um)",
    config.DEFAULT_SLIT_WIDTH_M / config.APERTURE_WIDTH_UM.display_to_si,
)
slit_length, slit_length_box = linked_float(
    config.APERTURE_HEIGHT_UM,
    "slit length (um)",
    config.DEFAULT_SLIT_LENGTH_M / config.APERTURE_HEIGHT_UM.display_to_si,
)
rectangle_width, rectangle_width_box = linked_float(
    config.APERTURE_WIDTH_UM,
    "width (um)",
    config.DEFAULT_RECTANGLE_WIDTH_M / config.APERTURE_WIDTH_UM.display_to_si,
)
rectangle_height, rectangle_height_box = linked_float(
    config.APERTURE_HEIGHT_UM,
    "height (um)",
    config.DEFAULT_RECTANGLE_HEIGHT_M / config.APERTURE_HEIGHT_UM.display_to_si,
)
circle_radius, circle_radius_box = linked_float(
    config.CIRCLE_RADIUS_UM,
    "radius (um)",
    config.DEFAULT_CIRCLE_RADIUS_M / config.CIRCLE_RADIUS_UM.display_to_si,
)

aperture_rows = []
aperture_rows_box = widgets.VBox()
add_aperture_button = widgets.Button(description="Add aperture", icon="plus")


def refresh_aperture_rows():
    aperture_rows_box.children = tuple(row["box"] for row in aperture_rows)


def add_aperture_row(kind="slit", center_mm=None):
    center_value = config.CENTER_MM.default if center_mm is None else center_mm
    shape = widgets.Dropdown(options=config.SHAPE_OPTIONS, value=kind, description="Shape")
    orientation = widgets.Dropdown(
        options=config.SLIT_ORIENTATION_OPTIONS,
        value="vertical",
        description="orientation",
    )
    width = bounded_float(config.APERTURE_WIDTH_UM, "width (um)")
    height = bounded_float(config.APERTURE_HEIGHT_UM, "height/length (um)")
    radius = bounded_float(config.CIRCLE_RADIUS_UM, "radius (um)")
    center_x = bounded_float(config.CENTER_MM, "x center (mm)", center_value)
    center_y = bounded_float(config.CENTER_MM, "y center (mm)")
    remove = widgets.Button(description="Remove", icon="trash")
    dimensions = widgets.HBox()
    row_box = widgets.VBox()
    row = {
        "shape": shape,
        "orientation": orientation,
        "width": width,
        "height": height,
        "radius": radius,
        "center_x": center_x,
        "center_y": center_y,
        "remove": remove,
        "box": row_box,
    }

    def refresh_dimensions(change=None):
        if shape.value == "circle":
            dimensions.children = (radius,)
        elif shape.value == "slit":
            dimensions.children = (width, height, orientation)
        else:
            dimensions.children = (width, height)
        row_box.children = (
            widgets.HBox([shape, remove]),
            dimensions,
            widgets.HBox([center_x, center_y]),
        )

    def remove_row(button):
        if row in aperture_rows:
            aperture_rows.remove(row)
            refresh_aperture_rows()

    shape.observe(refresh_dimensions, names="value")
    remove.on_click(remove_row)
    refresh_dimensions()
    aperture_rows.append(row)
    refresh_aperture_rows()


for initial_center in config.DEFAULT_MULTIPLE_CENTERS_MM:
    add_aperture_row("slit", initial_center)

add_aperture_button.on_click(lambda button: add_aperture_row())

case_controls = widgets.VBox()


def refresh_case_controls(change=None):
    if case.value == "slit":
        case_controls.children = (
            slit_orientation,
            slit_width_box,
            slit_length_box,
        )
    elif case.value == "rectangle":
        case_controls.children = (rectangle_width_box, rectangle_height_box)
    elif case.value == "circle":
        case_controls.children = (circle_radius_box,)
    else:
        case_controls.children = (add_aperture_button, aperture_rows_box)


case.observe(refresh_case_controls, names="value")
refresh_case_controls()


def build_apertures():
    if case.value == "slit":
        width = config.APERTURE_WIDTH_UM.to_si(slit_width.value)
        length = config.APERTURE_HEIGHT_UM.to_si(slit_length.value)
        if slit_orientation.value == "horizontal":
            width, length = length, width
        return [engine.Aperture.slit(width, length)]
    if case.value == "rectangle":
        return [
            engine.Aperture.rectangle(
                config.APERTURE_WIDTH_UM.to_si(rectangle_width.value),
                config.APERTURE_HEIGHT_UM.to_si(rectangle_height.value),
            )
        ]
    if case.value == "circle":
        return [engine.Aperture.circle(config.CIRCLE_RADIUS_UM.to_si(circle_radius.value))]

    apertures = []
    for row in aperture_rows:
        center_x = config.CENTER_MM.to_si(row["center_x"].value)
        center_y = config.CENTER_MM.to_si(row["center_y"].value)
        if row["shape"].value == "circle":
            apertures.append(
                engine.Aperture.circle(
                    config.CIRCLE_RADIUS_UM.to_si(row["radius"].value),
                    center_x,
                    center_y,
                )
            )
        elif row["shape"].value == "slit":
            width = config.APERTURE_WIDTH_UM.to_si(row["width"].value)
            length = config.APERTURE_HEIGHT_UM.to_si(row["height"].value)
            if row["orientation"].value == "horizontal":
                width, length = length, width
            apertures.append(
                engine.Aperture.slit(
                    width,
                    length,
                    center_x,
                    center_y,
                )
            )
        else:
            apertures.append(
                engine.Aperture.rectangle(
                    config.APERTURE_WIDTH_UM.to_si(row["width"].value),
                    config.APERTURE_HEIGHT_UM.to_si(row["height"].value),
                    center_x,
                    center_y,
                )
            )
    return apertures


def first_minima(apertures, lambda_0, z, n):
    if case.value in ("slit", "rectangle"):
        aperture = apertures[0]
        return engine.rectangle_first_minima(
            lambda_0, z, aperture.width, aperture.height, n
        )
    if case.value == "circle":
        radius = engine.circular_first_minimum(
            lambda_0, z, apertures[0].radius, n
        )
        return radius, radius
    return None


def active_views():
    return {
        view_key
        for view_key, button in view_buttons.items()
        if button.value
    }


def show_aperture_plane(apertures):
    """Draw the selected openings in the aperture plane."""
    millimetres = config.CENTER_MM.display_to_si
    support = engine.aperture_support_radius(apertures)
    limit = config.APERTURE_VIEW_MARGIN * support / millimetres
    colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
    figure, axis = plt.subplots(figsize=config.APERTURE_FIGURE_SIZE)

    for index, aperture in enumerate(apertures):
        color = colors[index % len(colors)]
        center = (aperture.center_x / millimetres, aperture.center_y / millimetres)
        label = f"{index + 1}: {aperture.kind.value}"
        if aperture.kind == engine.ApertureKind.CIRCLE:
            patch = Circle(
                center,
                aperture.radius / millimetres,
                facecolor=color,
                edgecolor=color,
                alpha=0.5,
                label=label,
            )
        else:
            patch = Rectangle(
                (
                    (aperture.center_x - aperture.width / 2.0) / millimetres,
                    (aperture.center_y - aperture.height / 2.0) / millimetres,
                ),
                aperture.width / millimetres,
                aperture.height / millimetres,
                facecolor=color,
                edgecolor=color,
                alpha=0.5,
                label=label,
            )
        axis.add_patch(patch)

    axis.set(
        xlim=(-limit, limit),
        ylim=(-limit, limit),
        aspect="equal",
        title="Aperture-plane shape",
        xlabel="x tilde (mm)",
        ylabel="y tilde (mm)",
    )
    axis.axhline(0.0, color="black", linewidth=0.6, alpha=0.4)
    axis.axvline(0.0, color="black", linewidth=0.6, alpha=0.4)
    axis.grid(alpha=0.2)
    axis.legend(loc="upper right")
    figure.tight_layout()
    plt.show()


def show_aperture_objective_geometry(apertures, z):
    """Draw aperture and observation planes in a normalized side view."""
    aperture_half = config.GEOMETRY_APERTURE_HALF_HEIGHT
    screen_half = config.GEOMETRY_SCREEN_HALF_HEIGHT
    vertical_limit = config.GEOMETRY_VERTICAL_LIMIT
    figure, axis = plt.subplots(figsize=config.GEOMETRY_FIGURE_SIZE)

    axis.axhline(0.0, color="black", linestyle=":", label="optical axis")
    axis.plot(
        [0.0, 0.0],
        [-aperture_half, aperture_half],
        linewidth=5.0,
        label="aperture plane",
    )
    axis.plot(
        [z, z],
        [-screen_half, screen_half],
        linewidth=5.0,
        label="objective / observation plane",
    )
    axis.plot([0.0, z], [aperture_half, screen_half], linestyle="--")
    axis.plot([0.0, z], [-aperture_half, -screen_half], linestyle="--")
    axis.annotate(
        f"z = {z:g} m",
        xy=(z / 2.0, 0.0),
        xytext=(z / 2.0, vertical_limit / 4.0),
        ha="center",
        arrowprops={"arrowstyle": "<->"},
    )
    axis.set(
        xlim=(-0.05 * z, 1.05 * z),
        ylim=(-vertical_limit, vertical_limit),
        title=(
            "Aperture-objective geometry "
            f"({len(apertures)} opening(s); transverse scale not to scale)"
        ),
        xlabel="propagation coordinate z (m)",
        ylabel="normalized transverse coordinate",
    )
    axis.legend(loc="upper left")
    axis.grid(alpha=0.2)
    figure.tight_layout()
    plt.show()


calculate_button = widgets.Button(
    description="Calculate pattern", icon="calculator", button_style="primary"
)
validity_status = widgets.HTML()
plot_output = widgets.Output()


def run_simulation(button=None):
    lambda_0 = config.WAVELENGTH_NM.to_si(wavelength.value)
    z = float(distance.value)
    n = float(refractive_index.value)
    selected = active_views()

    with plot_output:
        plot_output.clear_output(wait=True)
        if not selected:
            validity_status.value = "<b>Select at least one view.</b>"
            print("Use the view buttons to choose one or more outputs.")
            return
        try:
            apertures = build_apertures()
            result = screen.simulate_pattern(
                apertures,
                lambda_0,
                z,
                n=n,
                screen_half_width=config.SCREEN_HALF_WIDTH_MM.to_si(
                    screen_half_width.value
                ),
                resolution=resolution.value,
                zoom=zoom.value,
                max_fresnel_number=max_fresnel.value,
            )
        except engine.FarFieldError as error:
            report = error.report
            validity_status.value = (
                "<b>FAIL — outside the configured far field.</b> "
                f"N_F = {report.fresnel_number:.4g}; limit = "
                f"{report.max_fresnel_number:.4g}; required z >= "
                f"{report.required_distance:.4g} m."
            )
            print(str(error))
            return
        except (TypeError, ValueError) as error:
            validity_status.value = f"<b>Input error:</b> {error}"
            print(str(error))
            return

        report = result.far_field
        validity_status.value = (
            "<b>PASS — Fraunhofer calculation allowed.</b> "
            f"N_F = {report.fresnel_number:.4g} <= "
            f"{report.max_fresnel_number:.4g}; "
            f"lambda_medium = "
            f"{report.wavelength_medium / config.WAVELENGTH_NM.display_to_si:.3f} nm; "
            f"R_max = "
            f"{report.support_radius / config.APERTURE_WIDTH_UM.display_to_si:.3f} um."
        )

        profiles = screen.extract_central_profiles(
            result.X, result.Y, result.intensity
        )
        millimetres = config.SCREEN_HALF_WIDTH_MM.display_to_si
        extent = [
            result.X.min() / millimetres,
            result.X.max() / millimetres,
            result.Y.min() / millimetres,
            result.Y.max() / millimetres,
        ]

        figure, (pattern_axis, profile_axis) = plt.subplots(
            1,
            2,
            figsize=config.FIGURE_SIZE,
            gridspec_kw={"width_ratios": config.FIGURE_WIDTH_RATIOS},
        )
        image = pattern_axis.imshow(
            result.intensity,
            origin="lower",
            extent=extent,
            cmap=config.COLOR_MAP,
            aspect="equal",
        )
        colorbar = figure.colorbar(
            image, ax=pattern_axis, label="normalized intensity"
        )
        pattern_axis.set(
            title=f"{case.label} — observation plane",
            xlabel="x' (mm)",
            ylabel="y' (mm)",
        )

        profile_axis.plot(
            profiles.x / millimetres,
            profiles.horizontal,
            label="horizontal (y'=0)",
            color=config.PROFILE_COLORS[0],
        )
        profile_axis.plot(
            profiles.y / millimetres,
            profiles.vertical,
            label="vertical (x'=0)",
            color=config.PROFILE_COLORS[1],
        )
        profile_axis.set(
            title="Central intensity profiles",
            xlabel="screen coordinate (mm)",
            ylabel="normalized intensity",
        )
        profile_axis.grid(alpha=0.25)
        profile_axis.legend()

        minima = first_minima(apertures, lambda_0, z, n)
        if minima is not None:
            x_minimum, y_minimum = minima
            for position in (-x_minimum, x_minimum):
                pattern_axis.axvline(
                    position / millimetres,
                    color=config.PROFILE_COLORS[0],
                    linestyle="--",
                    alpha=0.65,
                )
                profile_axis.axvline(
                    position / millimetres,
                    color=config.PROFILE_COLORS[0],
                    linestyle="--",
                    alpha=0.45,
                )
            for position in (-y_minimum, y_minimum):
                pattern_axis.axhline(
                    position / millimetres,
                    color=config.PROFILE_COLORS[1],
                    linestyle="--",
                    alpha=0.65,
                )
                profile_axis.axvline(
                    position / millimetres,
                    color=config.PROFILE_COLORS[1],
                    linestyle=":",
                    alpha=0.45,
                )

        pattern_axis.set_visible("simulation" in selected)
        colorbar.ax.set_visible("simulation" in selected)
        profile_axis.set_visible("profiles" in selected)
        if "simulation" in selected or "profiles" in selected:
            figure.suptitle(
                f"lambda_0={wavelength.value:g} nm, n={n:g}, z={z:g} m, "
                f"N_F={report.fresnel_number:.3g}"
            )
            figure.tight_layout()
            plt.show()
        else:
            plt.close(figure)

        if "geometry" in selected:
            show_aperture_objective_geometry(apertures, z)
        if "aperture" in selected:
            show_aperture_plane(apertures)


calculate_button.on_click(run_simulation)
advanced_controls = widgets.Accordion(
    children=[
        widgets.VBox(
            [
                screen_half_width_box,
                zoom_box,
                resolution_box,
                max_fresnel_box,
            ]
        )
    ]
)
advanced_controls.set_title(0, "Advanced controls")

controls = widgets.VBox(
    [
        widgets.HTML("<b>Views (select any combination)</b>"),
        view_selector,
        case,
        wavelength_box,
        refractive_index_box,
        distance_box,
        case_controls,
        advanced_controls,
        calculate_button,
        validity_status,
    ]
)

# The reusable widget/aperture helpers above are assembled into the friendly
# dashboard in the next cell.

In [ ]:
# Friendly visual dashboard -------------------------------------------------

# Reuse the validated controls and aperture builders from the previous cell,
# but assemble them as a visual optical workbench.
case = widgets.ToggleButtons(
    options=config.DASHBOARD_CASE_OPTIONS,
    value="slit",
    description="Aperture",
    icons=("pause", "square-o", "circle-o", "th"),
    tooltips=("Single slit", "Rectangle", "Circle", "Multiple mixed openings"),
    style={"description_width": "initial"},
)
add_aperture_button = widgets.Button(
    description="Add opening", icon="plus", button_style="info"
)
auto_update = widgets.Checkbox(
    value=config.DEFAULT_AUTO_UPDATE,
    description="Auto update",
    indent=False,
)
refresh_button = widgets.Button(
    description="Refresh", icon="refresh", button_style="primary"
)
reset_button = widgets.Button(description="Reset", icon="undo")
extra_view_buttons = {
    key: widgets.ToggleButton(value=False, description=label)
    for label, key in config.EXTRA_VIEW_OPTIONS
}
secondary_boost, secondary_boost_box = linked_float(
    config.SECONDARY_MAXIMA_BOOST,
    "secondary maxima boost",
)

# Make the existing linked controls fit a compact, wrapping card.
_linked_controls = (
    (wavelength, wavelength_box),
    (refractive_index, refractive_index_box),
    (distance, distance_box),
    (slit_width, slit_width_box),
    (slit_length, slit_length_box),
    (rectangle_width, rectangle_width_box),
    (rectangle_height, rectangle_height_box),
    (circle_radius, circle_radius_box),
    (screen_half_width, screen_half_width_box),
    (zoom, zoom_box),
    (resolution, resolution_box),
    (max_fresnel, max_fresnel_box),
    (secondary_boost, secondary_boost_box),
)
for slider, linked_box in _linked_controls:
    slider.layout.width = config.DASHBOARD_SLIDER_WIDTH
    linked_box.layout = widgets.Layout(width="100%", flex_flow="row wrap")
    if len(linked_box.children) > 1:
        linked_box.children[1].layout.width = config.DASHBOARD_TEXT_WIDTH

case_controls.layout = widgets.Layout(width="100%")
aperture_rows_box.layout = widgets.Layout(width="100%")

schematic_output = widgets.Output(
    layout=widgets.Layout(width="100%", min_height="210px")
)
aperture_preview_output = widgets.Output(
    layout=widgets.Layout(width="100%", min_height="390px")
)
diffraction_preview_output = widgets.Output(
    layout=widgets.Layout(width="100%", min_height="390px")
)
extra_output = widgets.Output(layout=widgets.Layout(width="100%"))
validity_status = widgets.HTML()
spectrum_indicator = widgets.HTML()


def dashboard_card(title, children, width="100%"):
    card = widgets.VBox(
        [widgets.HTML(f"<div class='dashboard-card-title'>{title}</div>"), *children],
        layout=widgets.Layout(
            width=width,
            border=config.DASHBOARD_CARD_BORDER,
            padding=config.DASHBOARD_CARD_PADDING,
            margin=config.DASHBOARD_CARD_MARGIN,
        ),
    )
    card.add_class("dashboard-card")
    return card


def update_spectrum_indicator():
    span = config.WAVELENGTH_NM.maximum - config.WAVELENGTH_NM.minimum
    marker = 100.0 * (wavelength.value - config.WAVELENGTH_NM.minimum) / span
    spectrum_indicator.value = f"""
    <div class="spectrum-readout">
      <div><b>Vacuum wavelength:</b> {wavelength.value:g} nm</div>
      <div class="spectrum-track" style="background:{config.WAVELENGTH_GRADIENT_CSS};">
        <span class="spectrum-marker" style="left:{marker:.2f}%;"></span>
      </div>
    </div>
    """


def set_status(report=None, message=None):
    if message is not None:
        color = config.STATUS_NEUTRAL_COLOR
        text = message
    elif report.is_valid:
        color = config.STATUS_PASS_COLOR
        text = (
            f"FAR FIELD READY &nbsp; N<sub>F</sub>={report.fresnel_number:.3g} "
            f"&le; {report.max_fresnel_number:.3g} &nbsp; | &nbsp; "
            f"&lambda;<sub>medium</sub>="
            f"{report.wavelength_medium / config.WAVELENGTH_NM.display_to_si:.2f} nm"
        )
    else:
        color = config.STATUS_FAIL_COLOR
        text = (
            f"FRAUNHOFER BLOCKED &nbsp; N<sub>F</sub>={report.fresnel_number:.3g} "
            f"&gt; {report.max_fresnel_number:.3g} &nbsp; | &nbsp; "
            f"use z &ge; {report.required_distance:.3g} m"
        )
    validity_status.value = (
        f"<div class='status-badge' style='background:{color};'>{text}</div>"
    )


def clear_output_with_message(output, message):
    with output:
        output.clear_output(wait=True)
        print(message)


def draw_profiles(result, apertures, lambda_0, z, n):
    profiles = screen.extract_central_profiles(
        result.X, result.Y, result.intensity
    )
    millimetres = config.SCREEN_HALF_WIDTH_MM.display_to_si
    figure, axis = plt.subplots(figsize=config.DASHBOARD_PROFILE_FIGURE_SIZE)
    axis.plot(
        profiles.x / millimetres,
        profiles.horizontal,
        color=config.PROFILE_COLORS[0],
        label="horizontal (y'=0)",
    )
    axis.plot(
        profiles.y / millimetres,
        profiles.vertical,
        color=config.PROFILE_COLORS[1],
        label="vertical (x'=0)",
    )
    minima = first_minima(apertures, lambda_0, z, n)
    if minima is not None:
        for position in (-minima[0], minima[0]):
            axis.axvline(
                position / millimetres,
                color=config.PROFILE_COLORS[0],
                linestyle="--",
                alpha=0.45,
            )
        for position in (-minima[1], minima[1]):
            axis.axvline(
                position / millimetres,
                color=config.PROFILE_COLORS[1],
                linestyle=":",
                alpha=0.45,
            )
    axis.set(
        title="Central intensity profiles",
        xlabel="observation-plane coordinate (mm)",
        ylabel="normalized intensity",
    )
    axis.grid(alpha=0.25)
    axis.legend()
    figure.tight_layout()
    plt.show()


_rendering = False
_suspend_updates = False
_registered_rows = set()


def render_dashboard(button=None):
    global _rendering
    if _rendering:
        return
    _rendering = True
    update_spectrum_indicator()
    try:
        apertures = build_apertures()
        if not apertures:
            raise ValueError("Add at least one opening in Multiple mode.")
        lambda_0 = config.WAVELENGTH_NM.to_si(wavelength.value)
        z = float(distance.value)
        n = float(refractive_index.value)
        report = engine.evaluate_far_field(
            apertures,
            lambda_0,
            z,
            n=n,
            max_fresnel_number=max_fresnel.value,
        )
        set_status(report=report)

        with schematic_output:
            schematic_output.clear_output(wait=True)
            figure, axis = plt.subplots(figsize=config.SCHEMATIC_FIGURE_SIZE)
            dashboard.draw_optical_schematic(
                axis, apertures, z, wavelength.value, report
            )
            figure.tight_layout(pad=0.4)
            plt.show()

        with aperture_preview_output:
            aperture_preview_output.clear_output(wait=True)
            figure, axis = plt.subplots(
                figsize=config.DASHBOARD_PREVIEW_FIGURE_SIZE,
                facecolor=config.DASHBOARD_BACKGROUND,
            )
            dashboard.draw_aperture_preview(axis, apertures)
            figure.tight_layout(pad=0.5)
            plt.show()

        result = None
        with diffraction_preview_output:
            diffraction_preview_output.clear_output(wait=True)
            figure, axis = plt.subplots(
                figsize=config.DASHBOARD_PREVIEW_FIGURE_SIZE,
                facecolor=config.DASHBOARD_BACKGROUND,
            )
            if report.is_valid:
                result = screen.simulate_pattern(
                    apertures,
                    lambda_0,
                    z,
                    n=n,
                    screen_half_width=config.SCREEN_HALF_WIDTH_MM.to_si(
                        screen_half_width.value
                    ),
                    resolution=resolution.value,
                    zoom=zoom.value,
                    max_fresnel_number=max_fresnel.value,
                )
                dashboard.draw_diffraction_preview(
                    axis,
                    result.X,
                    result.Y,
                    result.intensity,
                    wavelength.value,
                    gamma=1.0 / secondary_boost.value,
                )
            else:
                dashboard.draw_far_field_unavailable(axis, report)
            figure.tight_layout(pad=0.5)
            plt.show()

        enabled_extras = {
            key for key, toggle in extra_view_buttons.items() if toggle.value
        }
        extras_card.layout.display = "" if enabled_extras else "none"
        with extra_output:
            extra_output.clear_output(wait=True)
            if "profiles" in enabled_extras:
                if result is None:
                    print("Profiles are unavailable until the far-field check passes.")
                else:
                    draw_profiles(result, apertures, lambda_0, z, n)
            if "geometry" in enabled_extras:
                show_aperture_objective_geometry(apertures, z)
    except (TypeError, ValueError) as error:
        set_status(message=f"Input error: {error}")
        clear_output_with_message(schematic_output, str(error))
        clear_output_with_message(aperture_preview_output, str(error))
        clear_output_with_message(diffraction_preview_output, str(error))
    finally:
        _rendering = False


def request_update(change=None):
    if _suspend_updates or not auto_update.value:
        return
    render_dashboard()


def register_aperture_row(row):
    row_id = id(row)
    if row_id in _registered_rows:
        return
    _registered_rows.add(row_id)
    for key in (
        "shape",
        "orientation",
        "width",
        "height",
        "radius",
        "center_x",
        "center_y",
    ):
        row[key].observe(request_update, names="value")
    row["remove"].on_click(lambda button: request_update())


def dashboard_add_aperture(button=None):
    add_aperture_row()
    register_aperture_row(aperture_rows[-1])
    request_update()


def dashboard_case_changed(change=None):
    refresh_case_controls(change)
    request_update()


def reset_dashboard(button=None):
    global _suspend_updates
    _suspend_updates = True
    try:
        case.value = "slit"
        slit_orientation.value = "vertical"
        wavelength.value = config.WAVELENGTH_NM.default
        refractive_index.value = config.REFRACTIVE_INDEX.default
        distance.value = config.DISTANCE_M.default
        slit_width.value = (
            config.DEFAULT_SLIT_WIDTH_M / config.APERTURE_WIDTH_UM.display_to_si
        )
        slit_length.value = (
            config.DEFAULT_SLIT_LENGTH_M / config.APERTURE_HEIGHT_UM.display_to_si
        )
        rectangle_width.value = (
            config.DEFAULT_RECTANGLE_WIDTH_M
            / config.APERTURE_WIDTH_UM.display_to_si
        )
        rectangle_height.value = (
            config.DEFAULT_RECTANGLE_HEIGHT_M
            / config.APERTURE_HEIGHT_UM.display_to_si
        )
        circle_radius.value = (
            config.DEFAULT_CIRCLE_RADIUS_M / config.CIRCLE_RADIUS_UM.display_to_si
        )
        screen_half_width.value = config.SCREEN_HALF_WIDTH_MM.default
        zoom.value = config.ZOOM.default
        resolution.value = int(config.RESOLUTION.default)
        max_fresnel.value = config.MAX_FRESNEL_NUMBER.default
        secondary_boost.value = config.SECONDARY_MAXIMA_BOOST.default
        for toggle in extra_view_buttons.values():
            toggle.value = False

        aperture_rows.clear()
        refresh_aperture_rows()
        for initial_center in config.DEFAULT_MULTIPLE_CENTERS_MM:
            add_aperture_row("slit", initial_center)
            register_aperture_row(aperture_rows[-1])
        refresh_case_controls()
    finally:
        _suspend_updates = False
    render_dashboard()


# Wire live updates. Float sliders already use continuous_update=False, so a
# simulation runs once on release rather than for every pointer movement.
case.observe(dashboard_case_changed, names="value")
for control in (
    wavelength,
    refractive_index,
    distance,
    slit_orientation,
    slit_width,
    slit_length,
    rectangle_width,
    rectangle_height,
    circle_radius,
    screen_half_width,
    zoom,
    resolution,
    max_fresnel,
    secondary_boost,
):
    control.observe(request_update, names="value")
for toggle in extra_view_buttons.values():
    toggle.observe(request_update, names="value")
auto_update.observe(
    lambda change: render_dashboard() if change["new"] else None,
    names="value",
)
for row in aperture_rows:
    register_aperture_row(row)
add_aperture_button.on_click(dashboard_add_aperture)
refresh_button.on_click(render_dashboard)
reset_button.on_click(reset_dashboard)
refresh_case_controls()

source_section = dashboard_card(
    "SOURCE",
    [spectrum_indicator, wavelength_box, refractive_index_box],
)
aperture_section = dashboard_card(
    "APERTURE",
    [case, case_controls],
)
propagation_section = dashboard_card(
    "PROPAGATION",
    [distance_box],
)
extra_selector = widgets.HBox(tuple(extra_view_buttons.values()))
display_section = dashboard_card(
    "DISPLAY",
    [
        screen_half_width_box,
        zoom_box,
        secondary_boost_box,
        widgets.Accordion(
            children=[widgets.VBox([resolution_box, max_fresnel_box])],
            titles=("Advanced sampling and validity",),
        ),
        widgets.HTML("<b>Additional views</b>"),
        extra_selector,
    ],
)
control_panel = dashboard_card(
    "CONTROLS",
    [source_section, aperture_section, propagation_section, display_section],
    width=config.DASHBOARD_CONTROL_WIDTH,
)

aperture_panel = dashboard_card(
    "SELECTED OPENING",
    [aperture_preview_output],
    width="100%",
)
diffraction_panel = dashboard_card(
    "CAMERA / OBSERVATION",
    [diffraction_preview_output],
    width="100%",
)
preview_column = widgets.VBox(
    [aperture_panel, diffraction_panel],
    layout=widgets.Layout(width="100%"),
)
schematic_panel = dashboard_card(
    "OPTICAL PATH",
    [schematic_output],
)
extras_card = dashboard_card("ANALYTICAL DETAILS", [extra_output])
extras_card.layout.display = "none"

header = widgets.HBox(
    [
        widgets.HTML(
            "<div class='dashboard-title'>Fraunhofer Diffraction Workbench"
            "<span>Analytical far-field simulator</span></div>"
        ),
        widgets.HBox([auto_update, refresh_button, reset_button]),
    ],
    layout=widgets.Layout(
        width="100%",
        justify_content="space-between",
        align_items="center",
        flex_flow="row wrap",
    ),
)
main_area = widgets.GridBox(
    [control_panel, preview_column],
    layout=widgets.Layout(
        width="100%",
        grid_template_columns=config.DASHBOARD_MAIN_COLUMNS,
        grid_gap=config.DASHBOARD_COLUMN_GAP,
        align_items="flex-start",
    ),
)
main_area.add_class("dashboard-main-grid")

style = widgets.HTML(
    f"""
    <style>
      .fraunhofer-dashboard {{ max-width: {config.DASHBOARD_MAX_WIDTH}; margin: 0 auto; }}
      .fraunhofer-dashboard .dashboard-card {{
        border-radius: 10px; background: #ffffff; box-shadow: 0 2px 8px rgba(16,24,40,.08);
        box-sizing: border-box;
      }}
      .fraunhofer-dashboard .dashboard-card-title {{
        color: #344054; font-size: 12px; font-weight: 700; letter-spacing: .08em;
        margin-bottom: 7px;
      }}
      .fraunhofer-dashboard .dashboard-title {{
        color: #101828; font-size: 23px; font-weight: 700; padding: 8px 4px;
      }}
      .fraunhofer-dashboard .dashboard-title span {{
        display: block; color: #667085; font-size: 12px; font-weight: 400;
      }}
      .fraunhofer-dashboard .status-badge {{
        color: white; border-radius: 6px; padding: 8px 12px; font-weight: 600;
        margin: 4px 6px 8px 6px;
      }}
      .fraunhofer-dashboard .spectrum-track {{
        position: relative; height: 16px; border-radius: 4px; margin: 7px 0;
        border: 1px solid #98a2b3;
      }}
      .fraunhofer-dashboard .spectrum-marker {{
        position: absolute; top: -4px; width: 3px; height: 24px; background: white;
        border: 1px solid #101828; box-shadow: 0 0 0 1px white;
      }}
      .fraunhofer-dashboard .widget-label {{ color: #344054; }}
      @media (max-width: 900px) {{
        .fraunhofer-dashboard .dashboard-main-grid {{
          grid-template-columns: 1fr !important;
        }}
        .fraunhofer-dashboard .dashboard-card {{ width: 100% !important; }}
      }}
    </style>
    """
)

friendly_dashboard = widgets.VBox(
    [style, header, validity_status, schematic_panel, main_area, extras_card],
    layout=widgets.Layout(width="100%"),
)
friendly_dashboard.add_class("fraunhofer-dashboard")
display(friendly_dashboard)
render_dashboard()

### Suggested experiments

- Increase slit width and verify that the horizontal central maximum becomes narrower ($x_{\min}=\lambda z/b$).
- Increase refractive index while holding vacuum wavelength fixed; the pattern contracts because $\lambda=\lambda_0/n$.
- Reduce distance until the validity gate blocks the calculation, then use the reported minimum distance.
- In mixed mode, start with two equal slits and vary their center separation to change the interference-fringe spacing.
- Change one mixed-mode row to **Circle** or **Rectangle** to observe coherent interference between different analytical aperture shapes.